# Unit 4: Music Genre Classifier - Hands-on Exercise

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

**Course**: [HuggingFace Audio Transformers Course - Unit 4](https://huggingface.co/learn/audio-course/chapter4)

## Certificate Exercise Requirements

- Fine-tune a model on [`marsyas/gtzan`](https://huggingface.co/datasets/marsyas/gtzan) dataset
- Achieve **>= 87% accuracy**
- Push the model to the HuggingFace Hub with the required kwargs

## Status: DONE ✅

Trained locally (distilHuBERT, 15 epochs) → **0.89 accuracy**. Model: [VoicesColeby/distilhubert-finetuned-gtzan](https://huggingface.co/VoicesColeby/distilhubert-finetuned-gtzan). Reproducible script: `scripts/train_unit4.py`.

## Setup

In [ ]:
!pip install -q transformers datasets evaluate accelerate librosa soundfile

# Login to HuggingFace Hub (required for pushing model)
from huggingface_hub import notebook_login
notebook_login()

## 1. Load and Explore the GTZAN Dataset

GTZAN contains 1000 audio tracks (30s each), 10 genres, 100 tracks per genre:
blues, classical, country, disco, hiphop, jazz, metal, pop, reggae, rock

In [ ]:
from datasets import load_dataset

gtzan = load_dataset("marsyas/gtzan", "all", trust_remote_code=True)
print(gtzan)
print(f"\nGenre labels: {gtzan['train'].features['genre'].names}")
print(f"\nSample: {gtzan['train'][0].keys()}")

In [ ]:
import IPython.display as ipd

# Listen to a sample
sample = gtzan['train'][0]
print(f"Genre: {gtzan['train'].features['genre'].int2str(sample['genre'])}")
print(f"Sampling rate: {sample['audio']['sampling_rate']}")
ipd.Audio(sample['audio']['array'], rate=sample['audio']['sampling_rate'])

## 2. Train/Test Split and Preprocessing

In [ ]:
# Split the dataset
gtzan = gtzan["train"].train_test_split(seed=42, shuffle=True, test_size=0.1)
print(gtzan)

In [ ]:
from transformers import AutoFeatureExtractor
from datasets import Audio

# Choose your model - experiment with different ones to beat 87%!
# Options to try:
#   - "ntu-spml/distilhubert"               (DistilHuBERT - lighter, ~83%)
#   - "facebook/wav2vec2-base"              (Wav2Vec2)
#   - "MIT/ast-finetuned-audioset-10-10-0.4593"  (Audio Spectrogram Transformer - often 90%+)

model_id = "ntu-spml/distilhubert"
model_name = model_id.split("/")[-1]

feature_extractor = AutoFeatureExtractor.from_pretrained(
    model_id, do_normalize=True, return_attention_mask=True
)
print(f"Expected sampling rate: {feature_extractor.sampling_rate}")

# Resample dataset to match model's expected sampling rate
gtzan = gtzan.cast_column("audio", Audio(sampling_rate=feature_extractor.sampling_rate))

In [ ]:
import numpy as np

max_duration = 30.0

def preprocess_function(examples):
    audio_arrays = [x["array"] for x in examples["audio"]]
    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=feature_extractor.sampling_rate,
        max_length=int(feature_extractor.sampling_rate * max_duration),
        truncation=True,
        return_attention_mask=True,
    )
    return inputs

gtzan_encoded = gtzan.map(
    preprocess_function,
    remove_columns=["audio", "file"],
    batched=True,
    batch_size=100,
    num_proc=1,
)

# Rename genre to label for the Trainer
gtzan_encoded = gtzan_encoded.rename_column("genre", "label")
print(gtzan_encoded)

## 3. Define the Model and Training

In [ ]:
from transformers import AutoModelForAudioClassification, TrainingArguments, Trainer
import evaluate

# Labels
labels = ["blues", "classical", "country", "disco", "hiphop", "jazz", "metal", "pop", "reggae", "rock"]
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}

# Load model for classification
model = AutoModelForAudioClassification.from_pretrained(
    model_id,
    num_labels=len(labels),
    label2id=label2id,
    id2label=id2label,
)

In [ ]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    """Computes accuracy on a batch of predictions"""
    predictions = np.argmax(eval_pred.predictions, axis=-1)
    return accuracy.compute(predictions=predictions, references=eval_pred.label_ids)

In [ ]:
training_args = TrainingArguments(
    output_dir=f"{model_name}-finetuned-gtzan",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    per_device_eval_batch_size=8,
    num_train_epochs=15,
    warmup_steps=100,
    logging_steps=5,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True,
    push_to_hub=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=gtzan_encoded["train"],
    eval_dataset=gtzan_encoded["test"],
    tokenizer=feature_extractor,
    compute_metrics=compute_metrics,
)

trainer.train()

## 4. Evaluate

In [ ]:
eval_results = trainer.evaluate()
print(f"\nAccuracy: {eval_results['eval_accuracy']:.4f}")
print(f"Target:   >= 0.87")
print(f"{'PASSED' if eval_results['eval_accuracy'] >= 0.87 else 'NEEDS IMPROVEMENT - try different hyperparameters or model'}")

## 5. Push to Hub (Required for Certificate)

In [ ]:
kwargs = {
    "dataset_tags": "marsyas/gtzan",
    "dataset": "GTZAN",
    "model_name": f"{model_name}-finetuned-gtzan",
    "finetuned_from": model_id,
    "tasks": "audio-classification",
}

trainer.push_to_hub(**kwargs)

## Tips for Reaching 87% Accuracy

If you're below the target, try:
1. **Different base model**: Audio Spectrogram Transformer (AST) often works well
2. **More epochs**: Try 15-20 epochs
3. **Learning rate**: Try 1e-5 or 3e-5
4. **Longer audio segments**: Ensure you're using the full 30s clips
5. **Data augmentation**: Add noise, time stretching, pitch shifting